In [25]:
from openai import OpenAI
from pymilvus import model as milvus_model
from pymilvus import MilvusClient
from tqdm import tqdm

import os
import warnings
import logging

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

logging.getLogger("pymilvus").setLevel(logging.ERROR)

api_key = os.getenv("DEEPSEEK_API_KEY")
if not api_key:
    print("Can not find deepseek api key")
    exit()
else:
    print("Api key read successfully")

deepseek_client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/v1"
)

embedding_model = milvus_model.DefaultEmbeddingFunction()

test_embedding = embedding_model.encode_queries(["Vector len"])[0]

embedding_dim = len(test_embedding)
print(embedding_dim)
print(test_embedding[:10])

milvus_client = MilvusClient(uri="./milvus_test_demo.db",
                            timeout=30,
                             wait_for_ready=True)
collection_name = "my_1st_rag_collection"

if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embedding_dim,
    metric_type="IP",
    consistency_level="Strong"
)

data = []
text_lines = [
    "What door can never be closed? A goal",
    "What water can never be drunk? Tears",
    "What book can no one finish reading? A last will",
    "What bird cannot fly in the sky? An ostrich"
]

doc_embeddings = embedding_model.encode_documents(text_lines)

for i, line in enumerate(tqdm(text_lines, desc="Creating embeddings")):
    data.append({"id": i, "vector": doc_embeddings[i], "text": line})

milvus_client.insert(collection_name=collection_name, data=data)

question = "不会在天上看到的鸟？"

search_res = milvus_client.search(
    collection_name=collection_name,
    data=embedding_model.encode_queries([question]),
    limit=5,
    search_params={"metric_type": "IP", "params":{}},
    output_fields=["text"]
)

import json

retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]

for i, res in enumerate(search_res[0]):
    print("\n===== 第", i+1, "条结果 =====")
    print("\nres 本身 =", res)          # 看完整结构
    print("\nres['entity'] =", res["entity"])  # 字典
    print("\nres['entity']['text'] =", res["entity"]["text"]) # 正确取文本
    print("\nres['distance'] =", res["distance"]) # 相似度分数

print(json.dumps(retrieved_lines_with_distances, indent=5, ensure_ascii=False))

context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)

SYSTEM_PROMPT = """
你是一个 AI 助手。你能够从提供的上下文段落片段中找到问题的答案。
"""

USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。
最后追加原始回答的中文翻译，并用 <translated> 和 </translated> 标签包裹。

<context>
{context}
</context>

<question>
{question}
</question>

<translated>
</translated>
"""

response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)

print(response.choices[0].message.content)

Api key read successfully
768
[-0.02750775 -0.01074684  0.02967544 -0.01778205 -0.01723714 -0.0605397
 -0.0135056   0.06475154  0.01001156  0.01850883]


Creating embeddings: 100%|██████████| 4/4 [00:00<00:00, 37117.73it/s]



===== 第 1 条结果 =====

res 本身 = {'id': 0, 'distance': 0.19645798206329346, 'entity': {'text': 'What door can never be closed? A goal'}}

res['entity'] = {'text': 'What door can never be closed? A goal'}

res['entity']['text'] = What door can never be closed? A goal

res['distance'] = 0.19645798206329346

===== 第 2 条结果 =====

res 本身 = {'id': 3, 'distance': 0.1504937708377838, 'entity': {'text': 'What bird cannot fly in the sky? An ostrich'}}

res['entity'] = {'text': 'What bird cannot fly in the sky? An ostrich'}

res['entity']['text'] = What bird cannot fly in the sky? An ostrich

res['distance'] = 0.1504937708377838

===== 第 3 条结果 =====

res 本身 = {'id': 2, 'distance': 0.12673154473304749, 'entity': {'text': 'What book can no one finish reading? A last will'}}

res['entity'] = {'text': 'What book can no one finish reading? A last will'}

res['entity']['text'] = What book can no one finish reading? A last will

res['distance'] = 0.12673154473304749

===== 第 4 条结果 =====

res 本身 = {'id': 1

I0430 13:57:00.187093   13701 chttp2_transport.cc:1369] unix:/tmp/tmpnw25p3ca_milvus_test_demo.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {grpc_status:14, http2_error:11}
E0430 13:57:00.187602   13701 chttp2_transport.cc:1401] unix:/tmp/tmpnw25p3ca_milvus_test_demo.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms
